# Task 4: Simple Fashion Visual Search

**Status: simplified code prepared; not executed.** Existing search artifacts
still belong to the previous contrastive encoder. No new retrieval scores or
runtime claims are available. See [the change record](../docs/TASK0_TASK4_SIMPLIFICATION.md).

Represent each image by a fixed vector, normalize its length, and rank gallery
images by cosine similarity (the dot product of normalized vectors). Compare
RGB thumbnails against HOG shape + HSV colour histograms. No CNN training,
contrastive loss, special sampler or pretrained weights are needed.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys
import time

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch  # Serialization only; retrieval uses NumPy on CPU.

from app.server.utils.handcrafted import DEFAULT_FEATURE_CONFIG
from scripts.preprocessing import SEED, task_frame
from scripts.retrieval import embed_images, retrieval_metrics

FEATURES = {
    'pixel': {'image_size': [12, 16]},
    'hog_hsv': dict(DEFAULT_FEATURE_CONFIG),
}

## 2. Gallery, validation queries and test queries

Training rows form the evaluation gallery. **Validation** queries select the
feature method; test queries are evaluated only after that choice is frozen.
Primary relevance means matching article type, not human-rated visual similarity.
These are the existing group-isolated partitions, not a new untouched holdout.

In [ ]:
train_df = task_frame('articleType', 'train')
validation_df = task_frame('articleType', 'validation')
test_df = task_frame('articleType', 'test')
for query_frame in (validation_df, test_df):
    assert set(train_df.group_key).isdisjoint(query_frame.group_key)
assert set(validation_df.group_key).isdisjoint(test_df.group_key)
pd.Series({'gallery': len(train_df), 'validation_queries': len(validation_df),
           'test_queries': len(test_df)})

## 3. Compare two fixed representations on validation

Pixel vectors capture colour and coarse layout. HOG captures local edge patterns;
HSV histograms summarize colour. Both use bilinear resizing and unit-length
vectors. Their shared definitions are in [search_features.py](../app/server/utils/search_features.py).

Select the highest validation precision@5. Exact ties use mean reciprocal rank,
then fewer feature dimensions, then the method name for deterministic ordering.
Precision@k is the matching fraction of k results; hit rate@k means at least one
match; recall@k divides matching results by **all relevant gallery items**.
MRR measures the rank of the first matching item. Query averages favor common
classes; class support and the qualitative examples matter too.

In [ ]:
gallery_features = {}
comparison_rows = []
for method, config in FEATURES.items():
    started = time.perf_counter()
    gallery = embed_images(train_df, method, config)
    queries = embed_images(validation_df, method, config)
    metrics = retrieval_metrics(queries, gallery,
                                validation_df.articleType, train_df.articleType)
    gallery_features[method] = gallery
    comparison_rows.append({
        'method': method, **metrics, 'dimensions': gallery.shape[1],
        'gallery_mib': gallery.nbytes / 1024**2,
        'feature_and_evaluation_seconds': time.perf_counter() - started,
    })
comparison = pd.DataFrame(comparison_rows).set_index('method')
display(comparison)
ranked = comparison.reset_index().sort_values(
    ['precision@5', 'mean_reciprocal_rank', 'dimensions', 'method'],
    ascending=[False, False, True, True],
)
SELECTED_METHOD = ranked.iloc[0]['method']
print('Frozen method before test evaluation:', SELECTED_METHOD)

## 4. Evaluate the selected method on test queries

The fixed method is evaluated once here. Do not change feature settings in response
to this table. This is a follow-up on the existing partitions; previous test
exposure must be disclosed in the final report.

In [ ]:
gallery_embeddings = gallery_features[SELECTED_METHOD]
query_embeddings = embed_images(test_df, SELECTED_METHOD, FEATURES[SELECTED_METHOD])
test_metrics = retrieval_metrics(query_embeddings, gallery_embeddings,
                                 test_df.articleType, train_df.articleType)
pd.Series(test_metrics, name='selected_method_test')

## 5. Attribute agreement and examples

In [ ]:
top_five = []
for query in query_embeddings:
    scores = gallery_embeddings @ query
    top_five.append(np.argsort(-scores, kind='stable')[:5])
top_five = np.asarray(top_five)
attribute_agreement = {}
for attribute in ('articleType', 'baseColour', 'gender', 'usage', 'subCategory'):
    query_values = test_df[attribute].to_numpy()[:, None]
    retrieved_values = train_df[attribute].to_numpy()[top_five]
    valid = (query_values != '') & (retrieved_values != '')
    attribute_agreement[attribute] = {
        'agreement': float((query_values == retrieved_values)[valid].mean()) if valid.any() else np.nan,
        'valid_pairs': int(valid.sum()),
    }
display(pd.DataFrame(attribute_agreement).T)

Attribute agreement is a secondary diagnostic; literal `NA` counts as a usage
label, while genuinely blank pairs are excluded and their support is reported.
The fixed panel below includes common and rare article types plus catalogue
examples. It is descriptive, not a curated demonstration of only good matches.

In [ ]:
support = train_df.articleType.value_counts()
common = test_df.loc[test_df.articleType.isin(support.head(5).index)].head(2)
rare = test_df.loc[test_df.articleType.map(support).le(49)].head(2)
panel = pd.concat([common, rare, test_df.sample(6, random_state=SEED)])
query_indices = panel.drop_duplicates('id').head(6).index
positions = test_df.index.get_indexer(query_indices)
fig, axes = plt.subplots(len(positions), 6, figsize=(12, 2.5 * len(positions)), squeeze=False)
for row, position in enumerate(positions):
    items = [test_df.iloc[position]]
    items.extend(train_df.iloc[index] for index in top_five[position])
    for column, item in enumerate(items):
        with Image.open(item.image_path) as image:
            axes[row, column].imshow(image.convert('RGB'))
        axes[row, column].set_title(('Query: ' if column == 0 else '') + item.articleType, fontsize=8)
        axes[row, column].axis('off')
plt.tight_layout()

## 6. Export a deployment gallery after evaluation

This section will replace the existing search artifacts **only when executed**.
The deployment gallery can include all usable labelled catalogue images after
evaluation; its overlap with test queries means it must not be used to report
held-out scores. `visual_search_history.csv` now stores the validation comparison,
not a neural training curve, because there is no training.

In [ ]:
deployment_gallery = pd.concat([train_df, validation_df, test_df], ignore_index=True)
deployment_gallery = deployment_gallery.drop_duplicates('id')
deployment_embeddings = embed_images(deployment_gallery, SELECTED_METHOD, FEATURES[SELECTED_METHOD])
checkpoint = {
    'model_type': 'fixed_feature_cosine', 'feature_type': SELECTED_METHOD,
    'feature_config': FEATURES[SELECTED_METHOD],
    'embedding_dim': deployment_embeddings.shape[1],
    'selection_metric': 'validation_precision@5',
    'validation_metrics': comparison.loc[SELECTED_METHOD].to_dict(),
    'retrieval_metrics': test_metrics, 'seed': SEED,
    'experiment': 'fixed_feature_search_v1',
}
model_dir = ROOT / 'models'
model_dir.mkdir(exist_ok=True)
np.save(model_dir / 'visual_search_embeddings.npy', deployment_embeddings)
deployment_gallery[['id', 'image_path', 'articleType', 'baseColour', 'gender', 'usage', 'subCategory']].to_csv(
    model_dir / 'visual_search_metadata.csv', index=False,
)
torch.save(checkpoint, model_dir / 'visual_search_model.pt')
comparison.to_csv(model_dir / 'visual_search_history.csv')
print('Deployment index MiB:', deployment_embeddings.nbytes / 1024**2)

## 7. Interpretation and later verification

**Pending execution:** no comparison, test score, latency, qualitative finding or
winning method is claimed yet. After the authorized run, interpret the recorded
tables and panel and verify the saved index through the existing search CLI/API.
The current API continues to load the previous neural artifacts until export.

This design is easier to explain and removes neural training. Its limits are
substantial: handcrafted appearance may miss semantic similarity, white catalogue
backgrounds can dominate pixel scores, raw HOG/HSV scaling fixes the balance
between shape and colour, and exact search scans the whole gallery. HOG vectors
also use more index memory than the old 128-dimensional encoder. Higher category
agreement alone would not establish better visual similarity or superiority to
the previous encoder; that comparison has not been run.